# Mapa de IPM por anos - Ciudad Paraiso

Mapa del Indice de Pobreza Multidimensional por manzana (corte unico censal) con un panel
que muestra la **evolucion temporal del IPM de Ciudad Paraiso (Comuna 3) 2019-2025** y su
tabla de referencia.

- Coropleta por manzana: `data/geojson/IPM_ciudad_Paraiso.geojson` (escala de 4 niveles,
  tipo calor). Manzanas sin dato -> gris.
- Serie temporal (comuna): `data/IPM/Estadisticas_Distritales_IPM_periodica.xlsx` (hoja
  Comuna = IPM por comuna/ano; Cali como referencia).
- Nota: el mapa por manzana es un corte unico (no cambia por ano); lo que varia por ano es
  el IPM global de la comuna (panel). El ano 2020 tiene base muy pequena (no representativo).


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
if _en_colab():
    import os, subprocess, sys, zipfile
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST):
        subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    _zip=os.path.join(DEST,'data_publico','data_itt_publico.zip')
    if os.path.isfile(_zip):
        with zipfile.ZipFile(_zip) as z: z.extractall(os.path.join(DEST,'data'))
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo:', os.getcwd())
else:
    print('Entorno local (uv). Setup Colab omitido.')


## 1. Entorno y paleta

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"
OKI_AMARILLO="#F0E442"; OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"
OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"
CRS_WEB=4326; CRS_METRIC=9377
print("Entorno listo.")

## 2. Rutas y poligono de la zona

In [ ]:
def _resolver_poligono():
    env=os.environ.get("POLIGONO_ZONA"); cand=[env] if env else []
    cand+=[os.path.join("..","data","geojson","poligono_ciudad_Paraiso.geojson"),
           os.path.join("data","geojson","poligono_ciudad_Paraiso.geojson"),
           "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
           "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson"]
    for p in cand:
        if p and os.path.isfile(p): return os.path.abspath(p)
    return None
_pp=_resolver_poligono()
POLIGONO_ZONA=gpd.read_file(_pp).to_crs(CRS_WEB) if _pp else None
print("Poligono:", _pp)
OUT_DIR=None
for _b in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_b,exist_ok=True); OUT_DIR=os.path.abspath(_b); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Salida de mapas HTML:", MAPAS_DIR)
def _ruta_itt(n):
    for p in [os.path.join("..","data","1_data_itt",n),os.path.join("data","1_data_itt",n),
              os.path.join("/content","data","1_data_itt",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","1_data_itt",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
def _ruta_geojson(n):
    for p in [os.path.join("..","data","geojson",n),os.path.join("data","geojson",n),
              os.path.join("/content","data","geojson",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","geojson",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
def _ruta_ipm_xlsx(n):
    for p in [os.path.join("..","data","IPM",n),os.path.join("data","IPM",n),
              os.path.join("/content","data","IPM",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","IPM",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None

## 3. Carga: IPM por manzana (corte unico) + serie temporal de la comuna

In [ ]:
# 3a. Coropleta IPM por manzana.
_rp=_ruta_geojson("IPM_ciudad_Paraiso.geojson")
if _rp is None:
    raise FileNotFoundError("No se encontro IPM_ciudad_Paraiso.geojson en data/geojson.")
ipm_z=gpd.read_file(_rp)
ipm_z=ipm_z.set_crs(epsg=CRS_WEB) if ipm_z.crs is None else ipm_z.to_crs(CRS_WEB)
ipm_z=ipm_z[ipm_z.geometry.notna()].copy()
ipm_z["ipm"]=pd.to_numeric(ipm_z["ipm"], errors="coerce")
if POLIGONO_ZONA is not None:
    _z=POLIGONO_ZONA.geometry.union_all()
    ipm_z=ipm_z[ipm_z.geometry.within(_z) | ipm_z.geometry.intersects(_z)].copy()
ipm_z=ipm_z.reset_index(drop=True)
print("Manzanas IPM en zona:", len(ipm_z), "| con dato:", int(ipm_z['ipm'].notna().sum()))

# 3b. Serie temporal del IPM por comuna (hoja Comuna) y Cali (hoja Anio).
def _serie_ipm():
    rp=_ruta_ipm_xlsx("Estadisticas_Distritales_IPM_periodica.xlsx")
    if rp is None:
        print("Aviso: no se encontro Excel distrital; serie vacia.")
        return [], []
    c=pd.read_excel(rp, sheet_name="Comuna", header=None)
    jc=[j for j,n in enumerate(c.iloc[0].tolist()) if isinstance(n,str) and n=="IPM"][0]
    d=c.iloc[3:].copy(); d[0]=d[0].ffill()
    d=d[pd.to_numeric(d[1],errors="coerce").notna()]
    d[0]=pd.to_numeric(d[0],errors="coerce").astype(int); d[1]=pd.to_numeric(d[1],errors="coerce").astype(int)
    c3=d[d[1]==3]
    serie=[]
    for _,r in c3.sort_values(0).iterrows():
        s=pd.to_numeric(r[jc],errors="coerce"); n=pd.to_numeric(r[jc+1],errors="coerce")
        serie.append((int(r[0]), round(100.0*s/n,2) if (pd.notna(s) and pd.notna(n) and n) else None))
    a=pd.read_excel(rp, sheet_name="Anio", header=None)
    ja=[j for j,n in enumerate(a.iloc[0].tolist()) if isinstance(n,str) and n=="IPM"][0]
    da=a.iloc[3:].copy(); da=da[pd.to_numeric(da[0],errors="coerce").notna()]; da[0]=da[0].astype(int)
    scali=[]
    for _,r in da.sort_values(0).iterrows():
        s=pd.to_numeric(r[ja],errors="coerce"); n=pd.to_numeric(r[ja+1],errors="coerce")
        scali.append((int(r[0]), round(100.0*s/n,2) if n else None))
    return serie, scali
SERIE_C3, SERIE_CALI = _serie_ipm()
print("Serie IPM Comuna 3:", SERIE_C3)
print("Serie IPM Cali:", SERIE_CALI)

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa (Folium)

In [ ]:
# 5a. Mapa IPM por manzana (corte unico) + panel con serie temporal de la comuna.
import folium
# Escala IPM de 4 niveles (Colombia urbano) tipo calor.
IPM_NIVELES = [
    (0.0, 8.0,   "Muy bajo / Bajo",       "#1a9850"),
    (8.1, 15.0,  "Medio / Tipico urbano", "#a6d96a"),
    (15.1, 25.0, "Alto",                  "#fdae61"),
    (25.1, 100.0,"Critico / Severo",      "#d73027"),
]
def _niv_idx(v):
    v = float(v)
    for i, (lo, hi, _n, _c) in enumerate(IPM_NIVELES):
        if v <= hi:
            return i
    return len(IPM_NIVELES) - 1
def _cat_nombre(v): return IPM_NIVELES[_niv_idx(v)][2]
def _color_ipm(v): return IPM_NIVELES[_niv_idx(v)][3]
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro = [_c.y, _c.x]
else:
    centro = [ipm_z.geometry.union_all().centroid.y, ipm_z.geometry.union_all().centroid.x]
m_ipm = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
                   zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m_ipm.get_root().width = "100%"; m_ipm.get_root().height = "100%"
m_ipm.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_ipm)
if POLIGONO_ZONA is not None:
    _zg = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]
    _ag = []
    for _gm in (list(_zg.geoms) if _zg.geom_type == "MultiPolygon" else [_zg]):
        _ag.append([[x, y] for x, y in _gm.exterior.coords])
    folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
        name="Mascara (foco en la zona)", control=False,
        style_function=lambda f: {"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.2}).add_to(m_ipm)
_rp_lim = _ruta_itt("00_limite_ciudad_paraiso.geojson")
_fg_limite = None
if _rp_lim is not None:
    _lim = gpd.read_file(_rp_lim).to_crs(CRS_WEB)
    _fg_limite = folium.FeatureGroup(name="Limite Ciudad Paraiso", show=True)
    folium.GeoJson(_lim.to_json(),
        style_function=lambda f: {"color":"#8A6D1A","weight":5,"dashArray":"10 6","fill":False}).add_to(_fg_limite)
# --- Capas del dataset ITT consolidado (data/1_data_itt), conmutables y apagadas ---
def _ruta_itt(nombre):
    cands = [
        __import__("os").path.join("..", "data", "1_data_itt", nombre),
        __import__("os").path.join("data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "drive", "MyDrive",
                   "mapa_Analitico_ciudad_Paraiso", "data", "1_data_itt", nombre),
    ]
    for _p in cands:
        if __import__("os").path.isfile(_p):
            return __import__("os").path.abspath(_p)
    return None
def _add_geojson_itt(nombre, capa_nombre, style, tooltip_field=None, popup_fn=None, to_points=False, show=False):
    _rp = _ruta_itt(nombre)
    if _rp is None:
        print("Aviso: no se encontro", nombre, "- capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Convertir columnas de fecha/timestamp a texto (folium no serializa Timestamp).
    for _col in _g.columns:
        if _col == "geometry":
            continue
        if str(_g[_col].dtype).startswith("datetime"):
            _g[_col] = _g[_col].astype(str)
    _fg = folium.FeatureGroup(name=capa_nombre, show=show)
    if to_points:
        for _, _r in _g.iterrows():
            if _r.geometry is None:
                continue
            _pt = _r.geometry
            folium.CircleMarker(
                location=[float(_pt.y), float(_pt.x)], radius=4,
                color="#333333", weight=1, fill=True, fill_color="#D55E00", fill_opacity=0.75,
                popup=(folium.Popup(popup_fn(_r), max_width=240) if popup_fn else None),
            ).add_to(_fg)
    else:
        folium.GeoJson(
            _g.to_json(),
            style_function=(lambda f, s=style: s),
            tooltip=(folium.GeoJsonTooltip(fields=[tooltip_field]) if tooltip_field and tooltip_field in _g.columns else None),
        ).add_to(_fg)
    print("Capa ITT agregada:", capa_nombre, "(", len(_g), "features )")
    return _fg
def _popup_hom300(r):
    f = r.get("fecha")
    try:
        f = f.strftime("%Y-%m-%d")
    except Exception:
        f = str(f)
    d = r.get("dist_a_estacion_m")
    arma = str(r.get("tipo_arma") or "s/d")
    movil = str(r.get("movil") or "s/d")
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
            '<div style="font-weight:700;color:#D55E00">Homicidio (&lt;=300 m estacion)</div>'
            '<div>Fecha: %s</div><div>Distancia: %s m</div>'
            '<div>Arma: %s</div><div>Movil: %s</div></div>'
            % (BG, f, (int(d) if d is not None else "s/d"), arma, movil))
# 01 manzanas con ITT consolidado: tooltip/popup con 4 atributos clave y area formateada.
def _add_manzanas_itt():
    _rp = _ruta_itt("01_manzanas_itt_consolidado.geojson")
    if _rp is None:
        print("Aviso: no se encontro manzanas consolidado - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Area metrica real en m2 (CRS metrico EPSG:9377). Si existe shape_area se ignora
    # para homogeneidad; se recalcula desde la geometria.
    _g["area_m2"] = _g.to_crs(CRS_METRIC).geometry.area
    # Formato legible: entero con separador de miles en punto (ej. 14.520 m2).
    _g["area_m2_fmt"] = _g["area_m2"].apply(lambda x: ("{:,}".format(int(round(x))).replace(",", ".")) + " m2")
    # Campos estandarizados.
    _g["id_manzana"] = _g["idmanzana"].astype(str)
    _g["barrio_f"] = _g["barrio"].astype(str)
    _g["manzana_f"] = _g["manzana"].astype(str)
    _fg = folium.FeatureGroup(name="ITT: manzanas (consolidado)", show=False)
    _tt = folium.GeoJsonTooltip(
        fields=["id_manzana", "barrio_f", "manzana_f", "area_m2_fmt"],
        aliases=["ID Manzana:", "Barrio:", "Manzana:", "Area:"],
        localize=True, sticky=True,
        style=("background-color:%s;color:%s;font-family:system-ui,Segoe UI,sans-serif;"
               "font-size:12px;padding:6px 8px;border-radius:6px;border:1px solid #ccc;" % (BG, OKI_GRIS)),
    )
    def _popup_mz(idm, bar, man, af):
        return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
                'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
                '<div style="font-weight:700;color:%s;margin-bottom:3px">Manzana %s</div>'
                '<table style="font-size:11px;border-collapse:collapse">'
                '<tr><td style="color:#666;padding:1px 6px">ID Manzana</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Barrio</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Manzana</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Area</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '</table></div>' % (BG, OKI_GRIS, man, idm, bar, man, af))
    folium.GeoJson(
        _g.to_json(),
        style_function=lambda f: {"fill": False, "color": "#333333",
                                  "weight": 1.0},
        tooltip=_tt,
    ).add_to(_fg)
    # Popups por manzana (tabla limpia).
    for _, _r in _g.iterrows():
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=lambda f: {"fillColor": "#56B4E9", "color": "#333333",
                                      "weight": 0.0, "fillOpacity": 0.0},
            popup=folium.Popup(_popup_mz(_r["id_manzana"], _r["barrio_f"],
                                         _r["manzana_f"], _r["area_m2_fmt"]), max_width=240),
        ).add_to(_fg)
    print("Capa ITT agregada: manzanas (consolidado) (", len(_g), "manzanas )")
    return _fg
# Demoliciones Cristo Rey: clasificar por estado de demolicion (columna 'demolicion').
DEMOL_COLOR = {
    "Demolido": "#D55E00",                       # bermellon: ya demolido
    "Programado para demolicion": "#E69F00",     # naranja: programado
    "En proceso (hacia demolicion)": "#F0E442",  # amarillo: en proceso
    "Sin demolicion programada": "#009E73",      # verde: conservado / sin intervencion
}
def _color_demol(v):
    return DEMOL_COLOR.get(str(v).strip(), "#999999")
def _popup_demol(r):
    est = str(r.get("demolicion") or "s/d")
    est2 = str(r.get("estado") or "")
    barrio = str(r.get("barrio") or "")
    npn = str(r.get("npn") or r.get("predio_catastral") or "")
    fdem = r.get("fecha_demolicion")
    try:
        fdem = fdem.strftime("%Y-%m-%d")
    except Exception:
        fdem = "" if fdem is None else str(fdem)
    filas = '<div style="font-weight:700;color:%s">Predio - %s</div>' % (_color_demol(est), est)
    if est2 and est2 != est:
        filas += '<div>Estado catastral: %s</div>' % est2
    if barrio:
        filas += '<div>Barrio: %s</div>' % barrio
    if npn:
        filas += '<div>Predio: %s</div>' % npn
    if fdem:
        filas += '<div>Fecha demolicion: %s</div>' % fdem
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:180px">%s</div>'
            % (BG, filas))
def _add_demoliciones():
    _rp = _ruta_itt("10_demoliciones_cristo_rey_predios.geojson")
    if _rp is None:
        print("Aviso: no se encontro demoliciones - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    for _c in _g.columns:
        if _c != "geometry" and str(_g[_c].dtype).startswith("datetime"):
            _g[_c] = _g[_c].astype(str)
    _fg = folium.FeatureGroup(name="ITT: demoliciones Cristo Rey", show=False)
    for _, _r in _g.iterrows():
        _col = _color_demol(_r.get("demolicion"))
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_col: {"fillColor": c, "color": "#333333",
                                               "weight": 0.8, "fillOpacity": 0.65}),
            tooltip="Demolicion: %s" % str(_r.get("demolicion") or "s/d"),
            popup=folium.Popup(_popup_demol(_r), max_width=260),
        ).add_to(_fg)
    print("Capa ITT agregada: demoliciones Cristo Rey (", len(_g), "predios )")
    return _fg
# --- Capas de referencia IPM e ICS por manzana (fuente oficial), apagadas ---
# Se agregan como contexto socioeconomico. Las leyendas de paleta aparecen SOLO cuando
# la capa correspondiente se activa en el control de capas (eventos overlayadd/remove).
def _add_ipm_ics_ref(_mapa):
    import os as _os
    _cands = [
        _os.path.join("..", "data", "geojson", "IPM_ciudad_Paraiso.geojson"),
        _os.path.join("data", "geojson", "IPM_ciudad_Paraiso.geojson"),
        _os.path.join("/content", "data", "geojson", "IPM_ciudad_Paraiso.geojson"),
        _os.path.join("/content", "drive", "MyDrive", "mapa_Analitico_ciudad_Paraiso",
                      "data", "geojson", "IPM_ciudad_Paraiso.geojson"),
    ]
    _rp = next((p for p in _cands if _os.path.isfile(p)), None)
    if _rp is None:
        print("Aviso: no se encontro IPM_ciudad_Paraiso.geojson - capas de referencia IPM/ICS omitidas")
        return (None, None)
    _g = gpd.read_file(_rp)
    _g = _g.set_crs(epsg=CRS_WEB) if _g.crs is None else _g.to_crs(CRS_WEB)
    _g = _g[_g.geometry.notna()].copy()
    _g["ipm"] = pd.to_numeric(_g["ipm"], errors="coerce")
    if "ics" in _g.columns:
        _g["ics"] = pd.to_numeric(_g["ics"], errors="coerce")
    if POLIGONO_ZONA is not None:
        _z = POLIGONO_ZONA.geometry.union_all()
        _g = _g[_g.geometry.within(_z) | _g.geometry.intersects(_z)].copy()
    if len(_g) == 0:
        print("Aviso: IPM/ICS sin manzanas en la zona - capas omitidas")
        return (None, None)
    # Escala oficial de 8 categorias (compartida) y paletas viridis (IPM) / cividis (ICS).
    _ESCALA = [(0.4,20.0,"Bajo"),(20.1,27.0,"Moderado-Bajo"),(27.1,33.1,"Moderado"),
               (33.2,39.8,"Moderado-Alto"),(39.9,48.6,"Alto"),(48.7,60.6,"Muy Alto"),
               (60.7,80.0,"Extremo"),(80.1,100.0,"Extremo Maximo")]
    _VIRIDIS = ["#440154","#472d7b","#3b528b","#2c728e","#21918c","#28ae80","#5ec962","#fde725"]
    _CIVIDIS = ["#00224e","#213d6b","#555b6c","#7d7c78","#a59c74","#c3b55f","#e1cc48","#fee838"]
    # IPM: escala de 4 niveles (Colombia urbano) tipo calor verde->rojo.
    _IPM_NIV = [(0.0,8.0,"Muy bajo / Bajo","#1a9850"),(8.1,15.0,"Medio / Tipico urbano","#a6d96a"),
                (15.1,25.0,"Alto","#fdae61"),(25.1,100.0,"Critico / Severo","#d73027")]
    def _ipm_idx(v):
        v = float(v)
        for _i,(lo,hi,_n,_c) in enumerate(_IPM_NIV):
            if v <= hi: return _i
        return len(_IPM_NIV)-1
    def _ipm_col(v): return _IPM_NIV[_ipm_idx(v)][3]
    def _ipm_nom(v): return _IPM_NIV[_ipm_idx(v)][2]
    def _idx(v):
        v = float(v)
        for _i,(lo,hi,_n) in enumerate(_ESCALA):
            if v <= hi: return _i
        return len(_ESCALA)-1
    def _nom(v): return _ESCALA[_idx(v)][2]
    _NOM_IPM = "Ref IPM: pobreza por manzana (" + str(len(_g)) + ")"
    _NOM_ICS = "Ref ICS: condicion social por manzana (" + str(len(_g)) + ")"
    _fg_ipm = folium.FeatureGroup(name=_NOM_IPM, show=False)
    for _, _r in _g.iterrows():
        _c = _ipm_col(_r["ipm"])
        folium.GeoJson(_r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_c: {"fillColor": c, "color": "#ffffff", "weight": 0.5, "fillOpacity": 0.75}),
            tooltip="IPM: %.1f (%s)" % (float(_r["ipm"]), _ipm_nom(_r["ipm"]))).add_to(_fg_ipm)
    _fg_ics = None
    if "ics" in _g.columns:
        _fg_ics = folium.FeatureGroup(name=_NOM_ICS, show=False)
        for _, _r in _g.iterrows():
            _c = _CIVIDIS[_idx(_r["ics"])]
            folium.GeoJson(_r["geometry"].__geo_interface__,
                style_function=(lambda f, c=_c: {"fillColor": c, "color": "#ffffff", "weight": 0.5, "fillOpacity": 0.6}),
                tooltip="ICS: %.1f (%s)" % (float(_r["ics"]), _nom(_r["ics"]))).add_to(_fg_ics)
    # Leyendas ocultas + toggle por overlayadd/overlayremove (aparecen solo con la capa activa).
    def _bloque(items):
        return "".join('<div style="display:flex;align-items:center;gap:5px;margin:1px 0">'
                       '<i style="width:14px;height:10px;background:%s;display:inline-block;border:1px solid #999"></i>'
                       '<span style="font-size:9px">%s</span></div>' % (col, nom) for (nom, col) in items)
    _it_ipm = _bloque([(n, c) for (_lo,_hi,n,c) in _IPM_NIV])
    _it_ics = _bloque([(n, c) for (_lo,_hi,n), c in zip(_ESCALA, _CIVIDIS)])
    _ley = (
        '<div id="ref-ley-ipm" style="display:none;position:fixed;bottom:24px;left:24px;z-index:9999;'
        'background:' + BG + ';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
        'font-family:system-ui,Segoe UI,sans-serif;color:' + OKI_GRIS + ';max-width:205px">'
        '<div style="font-weight:700;font-size:11px;margin-bottom:4px">IPM - Pobreza (viridis)</div>' + _it_ipm +
        '<div style="font-size:9px;color:#888;margin-top:3px">Mayor = mas pobreza</div></div>'
        '<div id="ref-ley-ics" style="display:none;position:fixed;bottom:24px;left:244px;z-index:9999;'
        'background:' + BG + ';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
        'font-family:system-ui,Segoe UI,sans-serif;color:' + OKI_GRIS + ';max-width:205px">'
        '<div style="font-weight:700;font-size:11px;margin-bottom:4px">ICS - Condicion social (cividis)</div>' + _it_ics +
        '<div style="font-size:9px;color:#888;margin-top:3px">Mayor = mejor condicion</div></div>'
        '<script>(function(){'
        'var MAP="m_mov", NI="' + _NOM_IPM + '", NC="' + _NOM_ICS + '";'
        'function hook(){var mp=window[MAP]; if(!mp){setTimeout(hook,300);return;}'
        'function tog(nombre,on){var id=null;'
        'if(nombre===NI) id="ref-ley-ipm"; else if(nombre===NC) id="ref-ley-ics";'
        'if(id){var el=document.getElementById(id); if(el) el.style.display=on?"block":"none";}}'
        'mp.on("overlayadd",function(e){tog(e.name,true);});'
        'mp.on("overlayremove",function(e){tog(e.name,false);});}'
        'if(document.readyState==="complete"){setTimeout(hook,400);}'
        'else{window.addEventListener("load",function(){setTimeout(hook,400);});}})();</script>'
    )
    _ley = _ley.replace("m_mov", _mapa.get_name())
    _mapa.get_root().html.add_child(folium.Element(_ley))
    print("Capas de referencia IPM/ICS agregadas (", len(_g), "manzanas ) con leyendas por activacion")
    return (_fg_ipm, _fg_ics)
# Capas transversales (apagadas; limite ya agregado arriba).
_fg_manzanas = _add_manzanas_itt()
_fg_estacion = _add_geojson_itt("11_estacion_fray_damian_manzana.geojson", "ITT: estacion Fray Damian",
                                {"color": "#0072B2", "weight": 2.75, "fill": False}, tooltip_field="nombre")
_fg_demol = _add_demoliciones()
_fg_ipm_ref, _fg_ics_ref = _add_ipm_ics_ref(m_ipm)
# Coropleta IPM por manzana (corte unico). ACTIVA. Manzanas sin dato -> gris.
def _popup_ipm(r):
    v = r.get("ipm")
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
                'background:%s;padding:6px 8px;border-radius:6px"><b>IPM manzana</b><br>Sin dato</div>' % BG)
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:180px">'
            '<div style="font-weight:700;color:#7a0177">IPM manzana</div>'
            '<div>Indice IPM: <b>%.1f</b></div><div>Nivel: <b>%s</b></div>'
            '<div style="font-size:10px;color:#888;margin-top:3px">Corte unico (censal). La serie por anio es de la comuna.</div></div>'
            % (BG, float(v), _cat_nombre(v)))
_fg_ipm = folium.FeatureGroup(name="IPM: indice por manzana (" + str(len(ipm_z)) + ")", show=True)
for _, r in ipm_z.iterrows():
    v = r.get("ipm")
    if v is None or (isinstance(v, float) and pd.isna(v)):
        _col = "#bdbdbd"; _tt = "IPM: sin dato"
    else:
        _col = _color_ipm(v); _tt = "IPM: %.1f (%s)" % (float(v), _cat_nombre(v))
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f, c=_col: {"fillColor": c, "color": "#ffffff", "weight": 0.6, "fillOpacity": 0.82}),
        tooltip=_tt, popup=folium.Popup(_popup_ipm(r), max_width=250)).add_to(_fg_ipm)
# Leyenda de paleta (4 niveles) + "Sin dato".
_items = ""
for (lo, hi, nom, col) in IPM_NIVELES:
    _rango = ("> %s" % ("%.0f" % (lo-0.1)).replace('.', ',')) if lo > 25 else "%s - %s" % (("%.1f" % lo).replace('.', ','), ("%.1f" % hi).replace('.', ','))
    _items += ('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
               '<i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>'
               '<span style="font-size:10px">%s (%s)</span></div>' % (col, nom, _rango))
_items += ('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
           '<i style="width:16px;height:12px;background:#bdbdbd;display:inline-block;border:1px solid #999"></i>'
           '<span style="font-size:10px">Sin dato</span></div>')
_leyenda = ('<div id="ipm-leyenda" style="position:fixed;bottom:24px;right:24px;z-index:9999;'
            'background:' + BG + ';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
            'font-family:system-ui,Segoe UI,sans-serif;color:' + OKI_GRIS + ';max-width:215px">'
            '<div style="font-weight:700;font-size:11px;margin-bottom:4px">IPM - Pobreza (nivel)</div>'
            + _items + '<div style="font-size:9px;color:#888;margin-top:4px">Mayor = mas pobreza. Umbral: 33,3</div></div>')
m_ipm.get_root().html.add_child(folium.Element(_leyenda))
_orden = [_fg_limite, _fg_manzanas, _fg_ipm_ref, _fg_ics_ref, _fg_estacion, _fg_ipm, _fg_demol]
for _capa in _orden:
    if _capa is not None:
        _capa.add_to(m_ipm)
if POLIGONO_ZONA is not None:
    m_ipm.fit_bounds([[3.442445,-76.53259],[3.44791,-76.526004]], padding=(30,30))
folium.LayerControl(collapsed=False, position="topright").add_to(m_ipm)
print("Mapa IPM (corte unico) construido. Manzanas:", len(ipm_z))

## 6. Panel: serie temporal del IPM (comuna) + tabla

In [ ]:
# 6a. Panel: serie temporal del IPM de Ciudad Paraiso (Comuna 3) 2019-2025 + tabla.
#      Grafico de linea (comuna) con Cali como referencia. Tabla de serie.
# SERIE_C3 y SERIE_CALI se cargan en la celda de datos (desde el Excel distrital).
_anios = [a for (a, _v) in SERIE_C3]
_c3 = [v for (_a, v) in SERIE_C3]
_cali = [dict(SERIE_CALI).get(a) for a in _anios]
# Marcar 2020 como no representativo (base muy pequena).
_nota2020 = any(a == 2020 for a in _anios)
import json as _json
_chart = _json.dumps({"anios": _anios, "c3": _c3, "cali": _cali}, ensure_ascii=False)
# Tabla de referencia.
_filas = ""
for a, v in SERIE_C3:
    _cal = dict(SERIE_CALI).get(a)
    _marca = " *" if a == 2020 else ""
    _filas += ('<tr><td style="padding:1px 6px">%d%s</td>'
               '<td style="padding:1px 6px;text-align:right"><b>%s</b></td>'
               '<td style="padding:1px 6px;text-align:right;color:#666">%s</td></tr>'
               % (a, _marca, ("%.1f" % v).replace('.', ',') if v is not None else "s/d",
                  ("%.1f" % _cal).replace('.', ',') if _cal is not None else "s/d"))
_tabla = ('<table style="font-size:11px;border-collapse:collapse;width:100%">'
          '<tr style="border-bottom:1px solid #ccc">'
          '<th style="padding:1px 6px;text-align:left">Ano</th>'
          '<th style="padding:1px 6px;text-align:right">Cdad Paraiso</th>'
          '<th style="padding:1px 6px;text-align:right">Cali</th></tr>'
          + _filas + '</table>')
_resumen = ('<div style="margin:2px 0 4px 0">Manzanas (corte unico): <b>%d</b> &nbsp;|&nbsp; '
            'IPM 2025 comuna: <b>%s</b></div>'
            % (len(ipm_z), ("%.1f" % dict(SERIE_C3).get(max(_anios))).replace('.', ',')))
_panel = """
<div id="ipm-panel" style="position:fixed;top:20px;left:20px;z-index:9999;
     background:__BG__;max-width:440px;width:min(440px,44vw);
     padding:10px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;margin-bottom:4px">IPM - Serie temporal (Ciudad Paraiso)</div>
  __RESUMEN__
  <div style="font-size:9px;color:#666;margin:2px 0 6px">El mapa es un corte unico por manzana; la serie es el IPM de la Comuna 3 por ano (fuente distrital). __NOTA2020__</div>
  <div style="position:relative;height:190px;width:100%"><canvas id="ipm-serie"></canvas></div>
  <details style="margin-top:8px" open>
    <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">Tabla de serie / Ocultar</summary>
    <div style="margin-top:6px;max-height:200px;overflow:auto">__TABLA__
      <div style="font-size:9px;color:#888;margin-top:3px">* 2020: base muy pequena, no representativo.</div>
    </div>
  </details>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var D = __CHART__;
  function dibujar(){
    var cv=document.getElementById('ipm-serie'); if(!cv||typeof Chart==='undefined')return;
    new Chart(cv.getContext('2d'),{type:'line',
      data:{labels:D.anios,datasets:[
        {label:'Ciudad Paraiso (Comuna 3)',data:D.c3,borderColor:'#d73027',backgroundColor:'#d73027',borderWidth:2,pointRadius:3,tension:0.2},
        {label:'Cali (referencia)',data:D.cali,borderColor:'#999',backgroundColor:'#999',borderWidth:2,borderDash:[5,4],pointRadius:2,tension:0.2}]},
      options:{responsive:true,maintainAspectRatio:false,interaction:{mode:'index',intersect:false},
        plugins:{legend:{labels:{boxWidth:10,font:{size:9}}}},
        scales:{x:{ticks:{font:{size:9}}},
                y:{beginAtZero:true,title:{display:true,text:'IPM (% hogares pobres)',font:{size:9}},ticks:{font:{size:9}}}}}});
  }
  if(document.readyState==='complete'){setTimeout(dibujar,300);}
  else{window.addEventListener('load',function(){setTimeout(dibujar,300);});}
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS).replace("__AZUL__", OKI_AZUL)
                .replace("__RESUMEN__", _resumen).replace("__TABLA__", _tabla)
                .replace("__CHART__", _chart)
                .replace("__NOTA2020__", "El ano 2020 tiene base muy pequena (no representativo)." if _nota2020 else ""))
m_ipm.get_root().html.add_child(folium.Element(_panel))
print("Panel de serie temporal IPM inyectado.")

## 7. Exportacion del HTML

In [ ]:
agregar_export(m_ipm, "IPM por anos - Ciudad Paraiso", "ipm_series")
html=os.path.join(MAPAS_DIR,"mapa_ipm_series.html")
m_ipm.save(html)
print("Mapa IPM series exportado:", html)
m_ipm